<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png"  width='15%'>  


<h1> Demo: LLM Indexing - Getting Started with FAISS</h1>
Developed by WeCloudData
<br></br>


# Importing required libraries

In [2]:
!nvidia-smi #to review that notebook has GPU access

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
%pip install PyMuPDF faiss-cpu sentence_transformers transformers pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import pandas as pd

c:\Users\alsae\Documents\SDA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




```
# This is formatted as code
```

## Build Document Vectors

In [34]:
import fitz

doc = fitz.open("ntp_en_annual_report_2025.pdf")
raw_text = ""

for page in doc:
    raw_text += page.get_text("text") + " "

cleaned_text = " ".join(raw_text.split())

import re

def fix_ligatures(text):
    return (text
        .replace("ﬀ", "ff")
        .replace("ﬁ", "fi")
        .replace("ﬂ", "fl")
        .replace("ﬃ", "ffi")
        .replace("ﬄ", "ffl")
        .replace("/f_", "f")  # FIX: restore missing letter
    )

cleaned_text = fix_ligatures(cleaned_text)

print(f"Total characters extracted: {len(cleaned_text)}")


Total characters extracted: 84764


In [80]:
words = cleaned_text.split()
chunks = []
current = []

for w in words:
    current.append(w)

    # If the word ends with a full stop, break here
    if w.endswith('.'):
        chunks.append(" ".join(current))
        current = []
        continue

    # If length exceeded, break before last word
    if len(" ".join(current)) > 500:
        chunks.append(" ".join(current[:-1]))
        current = [w]

# Add any remaining words
if current:
    chunks.append(" ".join(current))

In [81]:
print(chunks[0])

| Annual Report 2025 1 Introduction | Our Earth | Our Society | Our Economy | Our Future | KPIs | Closing Remarks | Annual Report 2025 2 Introduction | Our Earth | Our Society | Our Economy | Our Future | KPIs | Closing Remarks | Annual Report 2025 3 Introduction | Our Earth | Our Society | Our Economy | Our Future | KPIs | Closing Remarks | Annual Report 2025 4 Introduction | Our Earth | Our Society | Our Economy | Our Future | KPIs | Closing Remarks NTP has built a model for sustainable


In [82]:
from sentence_transformers import SentenceTransformer
# initialize sentence transformer model
model = SentenceTransformer('all-mpnet-base-v2', device = "cpu")
model.to("cpu")
# create sentence embeddings
chunk_embeddings = model.encode(chunks)
chunk_embeddings.shape

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5515.87it/s]


(300, 768)

## Faiss

### 1. IndexFlatL2

In [83]:
d = chunk_embeddings.shape[1]

In [84]:
index = faiss.IndexFlatL2(d)

In [85]:
index.is_trained

True

In [86]:
index.add(chunk_embeddings)

In [87]:
index.ntotal

300

In [88]:
k = 2
xq = model.encode(["That same evening"])

In [89]:
%%time
D, I = index.search(xq, k)  # search
print(I)

[[107  92]]
CPU times: total: 0 ns
Wall time: 1.92 ms


In [90]:
[chunks[i] for i in I[0]]

['from the project’s inception.',
 "Launch of NTP's 2 Phase nd Launch of the 1 Phase st •Launch of the National Award for Voluntary Work •Launch of the “Volunteer your Experience” initiative •Launch of “HRSD Volunteers” initiative •Launch of the “Shadow Coach” Program •Launch of the Social Responsibility Platform •Launch of the second Speakers Program •Modification of the flexible work policy •Issuance of Fundraising Law •Completion of the Parallel Training Program •Launch of “Ma’ak” App •Launch of “Ajeer” Service •Launch of the"]

If we’d rather extract the numerical vectors from Faiss, we can do that too.

In [ ]:

import numpy as np
vecs = np.zeros((k, d))
# then iterate through each ID from I and add the reconstructed vector to our zero-array
for i, val in enumerate(I[0].tolist()):
    vecs[i, :] = index.reconstruct(val)

In [92]:
vecs

array([[ 0.00563397,  0.07591618,  0.00202014, ..., -0.02495992,
        -0.0602014 , -0.02673293],
       [ 0.00925088,  0.03985523, -0.00738995, ..., -0.00528668,
        -0.01805961,  0.01380661]], shape=(2, 768))

### 2. FLV

In [93]:
d = chunk_embeddings.shape[1]

In [94]:
nlist = 2
quantizer = faiss.IndexFlatL2(d)
index = faiss.IndexIVFFlat(quantizer, d, nlist)

Here we’ve added a new parameter nlist. We use nlist to specify how many partitions (Voronoi cells) we’d like our index to have.

In [95]:
index.is_trained

False

Now that we added clustering with IndexIVFFlat, we will need to train the index. So, what we do now is train our index on our data — which we must do before adding any data to the index.



In [96]:
index.train(chunk_embeddings)
index.is_trained  # check if index is now trained

True

Now that our index is trained, we add our data just as we did before.


In [97]:
index.add(chunk_embeddings)
index.ntotal  # number of embeddings indexed

300

In [98]:
k = 2
xq = model.encode(["That same evening"])

In [99]:
%%time
D, I = index.search(xq, k)  # search
print(I)

[[107  92]]
CPU times: total: 0 ns
Wall time: 632 μs


The search time has clearly decreased, in this case, we don’t find any difference between results returned by our exhaustive search, and this approximate search. But, often this can be the case.



If approximate search with IndexIVFFlat returns suboptimal results, we can improve accuracy by increasing the search scope. We do this by increasing the nprobe attribute value — which defines how many nearby cells to search.

>

In [100]:
index.nprobe = 10

In [101]:
%%time
D, I = index.search(xq, k)  # search
print(I)

[[107  92]]
CPU times: total: 0 ns
Wall time: 661 μs


### RAG


1.   Split the original vector into several subvectors.
2.   For each set of subvectors, perform a clustering operation — creating multiple centroids for each sub-vector set.
3.   In the vector of sub-vectors, replace each sub-vector with the ID of it’s nearest set-specific centroid.






**RETRIEVAL FUNCTION**

In [ ]:

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def retrieve(query, top_k=2):
    query_emb = model.encode([query])
    distances, indices = index.search(query_emb, top_k)
    # Combines text from retrieved chunks correctly instead of just extracting lists
    retrieved_chunks = [chunks[i] for i in indices[0]]
    return retrieved_chunks 

**GENERATION MODEL (QA or Summarization)**

In [76]:
%pip install accelerate

Note: you may need to restart the kernel to use updated packages.


In [ ]:

model_id = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# AutoModelForSeq2SeqLM explicitly handles the text-to-text (encoder-decoder) format
t2t_model = AutoModelForSeq2SeqLM.from_pretrained(model_id, device_map="auto")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4640.59it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


**RAG PIPELINE FUNCTION**

In [ ]:

def rag_pipeline(query):
    retrieved_chunks = retrieve(query)
    
    # FIX: Join chunks into a continuous string before feeding to the prompt
    context = " ".join(retrieved_chunks) 
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"

    # Native text-to-text execution step
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(t2t_model.device)
    outputs = t2t_model.generate(**inputs, max_new_tokens=250, min_new_tokens=60, temperature=0.3, do_sample=True)
    
    # Decodes output back to a normal text string
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("🔍 Retrieved Chunks:\n", retrieved_chunks)
    print("\n💬 Model Answer:\n", response)

In [105]:
# rag_pipeline("What is the aim of the National Transformation Program?")
rag_pipeline("What is the aim of The National Transformation Program?")

🔍 Retrieved Chunks:
 ['To achieve this objective, the National Transformation Program leads a range of initiatives, including the digitization of government services, the promotion of e-commerce, and the expansion of internet connectivity across the Kingdom.', "H.E Mohammed bin Mazyed Al-Tuwaijri Member of the Council of Economic and Development Affairs Chairman of NTP's Committee | Annual Report 2025 7 Introduction | Our Earth | Our Society | Our Economy | Our Future | KPIs | Closing Remarks The National Transformation Program (NTP) aims to develop the necessary infrastructure and create an environment that enables the public, private, and non-profit sectors."]

💬 Model Answer:
 to develop the necessary infrastructure and create an environment that enables the public, private, and non-profit sectors. The National Transformation Program (NTP) aims to develop the necessary infrastructure and create an environment that enables the public, private, and non-profit sectors. The National Tra